**1. Sambungkan ke Google Drive**

In [1]:
from google.colab import drive
import os

# Mount Drive dengan paksa remount supaya refresh
drive.mount('/content/drive', force_remount=True)

# Path Folder
save_path = "/content/drive/MyDrive/SKRIPSI/CODE/CRAWLING DATA/HASIL_CRAWLING_MBG"

# Kita buat foldernya (tapi kalau sudah ada, dia diam saja/tidak error)
os.makedirs(save_path, exist_ok=True)

# Cek konfirmasi akhir
if os.path.exists(save_path):
    print(f"✅ SIP! Folder siap digunakan: {save_path}")
else:
    print(f"❌ Masih gagal menemukan folder. Cek nama folder di Drive kamu!")

Mounted at /content/drive
✅ SIP! Folder siap digunakan: /content/drive/MyDrive/SKRIPSI/CODE/CRAWLING DATA/HASIL_CRAWLING_MBG


**2. Instalasi Node.js & Tweet Harvest**

In [2]:
!sudo apt-get update
!sudo apt-get install -y curl
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!npx playwright install
!npx playwright install-deps
!npm install -g tweet-harvest

print("✅ Instalasi Selesai! Lanjut ke Cell berikutnya.")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.8 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,863 

**3. Set Token**

In [3]:
import os

# Ganti dengan token aslimu
my_token = "PASTE_YOUR_AUTH_TOKEN_HERE"

os.environ["TWITTER_AUTH_TOKEN"] = my_token

if my_token == "MASUKKAN_TOKEN_DISINI" or my_token == "":
    print("❌ Token belum diisi!")
else:
    print("✅ Token siap.")

✅ Token siap.


**4. Crawling**

In [7]:
# @title
import os
import time
import random
import pandas as pd
import glob
import shutil

# --- GANTI TANGGAL DI SINI (BATCH 1) ---
TARGET_START = "2025-03-15"
TARGET_END   = "2025-04-01"
PHASE_NAME   = "Maret"

# ---------------------------------------------
keywords = [
    "makan bergizi gratis",
    "\"makan bergizi gratis\"",
    "program MBG",
    "\"program MBG\"",
    "MBG",
    "\"MBG\"",
    "#MakanBergiziGratis",
    "#ProgramMBG",
    "#MBG"
]
limit = 500

# Path Folder Drive
save_path = "/content/drive/MyDrive/SKRIPSI/CODE/CRAWLING DATA/HASIL_CRAWLING_MBG"

print(f"🚀 MULAI CRAWLING: {TARGET_START} s.d {TARGET_END}")
print("="*60)

# 1. Bersihkan folder sementara
if os.path.exists("tweets-data"):
    shutil.rmtree("tweets-data")
os.makedirs("tweets-data", exist_ok=True)

# 2. Loop Crawling
files_to_merge = []

for kw in keywords:
    clean_kw = kw.replace('"', '').replace(' ', '_').replace('#', '')
    temp_csv = f"{clean_kw}.csv"

    search_query = f'{kw} since:{TARGET_START} until:{TARGET_END} lang:id'
    print(f"🔍 Mengambil: '{kw}' ...", end=" ")

    # Run Tweet Harvest
    command = (
        f'npx tweet-harvest -o "{temp_csv}" -s "{search_query}" -l {limit} --token "$TWITTER_AUTH_TOKEN"'
    )

    # Jalankan perintah
    os.system(command + " > /dev/null 2>&1")

    # --- BAGIAN PENGECEKAN (Pastikan masuk dalam indentasi FOR) ---
    full_temp_path = os.path.join("tweets-data", temp_csv)

    if os.path.exists(full_temp_path):
        if os.path.getsize(full_temp_path) > 0:
            files_to_merge.append(full_temp_path)
            print("✅ Dapat!")
        else:
            print("⚠️ Kosong (Kena Limit/Zonk)")
    else:
        print("❌ Gagal (File tidak tercipta)")

    # JEDA ANTI-LIMIT
    time.sleep(random.randint(15, 25))

# 3. GABUNG & SIMPAN
print("-" * 60)
if len(files_to_merge) > 0:
    print(f"📚 Menggabungkan {len(files_to_merge)} file...")

    all_dfs = []
    for f in files_to_merge:
        try:
            df = pd.read_csv(f)
            all_dfs.append(df)
        except Exception as e:
            print(f"⚠️ Gagal baca file {f}: {e}")

    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        final_name = f"DATA_{PHASE_NAME}_{TARGET_START}_{TARGET_END}.csv"
        final_path = os.path.join(save_path, final_name)

        os.makedirs(save_path, exist_ok=True)
        final_df.to_csv(final_path, index=False)

        print(f"🎉 SUKSES! Data tersimpan di Drive.")
        print(f"📂 Nama File: {final_name}")
        print(f"📊 Total Baris: {len(final_df)}")
    else:
        print("❌ Tidak ada data valid untuk digabung.")
else:
    print("❌ ZONK. Tidak ada tweet sama sekali.")

print("="*60)

🚀 MULAI CRAWLING: 2025-03-15 s.d 2025-04-01
🔍 Mengambil: 'makan bergizi gratis' ... ✅ Dapat!
🔍 Mengambil: '"makan bergizi gratis"' ... ✅ Dapat!
🔍 Mengambil: 'program MBG' ... ✅ Dapat!
🔍 Mengambil: '"program MBG"' ... ✅ Dapat!
🔍 Mengambil: 'MBG' ... ✅ Dapat!
🔍 Mengambil: '"MBG"' ... ✅ Dapat!
🔍 Mengambil: '#MakanBergiziGratis' ... ✅ Dapat!
🔍 Mengambil: '#ProgramMBG' ... ✅ Dapat!
🔍 Mengambil: '#MBG' ... ✅ Dapat!
------------------------------------------------------------
📚 Menggabungkan 9 file...
🎉 SUKSES! Data tersimpan di Drive.
📂 Nama File: DATA_Maret_2025-03-15_2025-04-01.csv
📊 Total Baris: 629
